In [1]:
!mkdir -p ~/work/transformer_chatbot/data/spa-eng

In [2]:
!python3 -m pip install --upgrade pip

In [3]:
!python3 -m pip install konlpy

In [4]:
!pip install mecab-python3

In [5]:
from konlpy.tag import Mecab

mecab = Mecab()
text = "LMS 환경에서도 코엔엘파이가 잘 작동합니다."
print(mecab.pos(text))

[('LMS', 'SL'), ('환경', 'NNG'), ('에서', 'JKB'), ('도', 'JX'), ('코', 'NNG'), ('엔', 'JKB+JX'), ('엘파', 'NNG'), ('이', 'NP'), ('가', 'JKS'), ('잘', 'MAG'), ('작동', 'NNG'), ('합니다', 'XSV+EF'), ('.', 'SF')]


In [6]:
!pip install sentencepiece nltk

In [7]:
import numpy as np
import pandas as pd
import torch
import sentencepiece as spm
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

import re
import os
import random
import math

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(torch.__version__)

2.7.1+cu118


In [8]:
import urllib.request
import zipfile

zip_filename = "spa-eng.zip"
zip_url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"

urllib.request.urlretrieve(zip_url, zip_filename)

with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(os.path.dirname(zip_filename))

print("슝=3")

슝=3


In [9]:
extracted_folder = "./spa-eng"
file_path = os.path.join(extracted_folder, "spa.txt")

with open(file_path, "r") as f:
    spa_eng_sentences = f.read().splitlines()

spa_eng_sentences = list(set(spa_eng_sentences))
total_sentence_count = len(spa_eng_sentences)
print("Example:", total_sentence_count)

for sen in spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Example: 118964
>> You're drunk.	Estás borracha.
>> Tom doesn't care about what Mary says about him.	A Tom no le importa lo que Mary diga acerca de él.
>> Thank you for the help.	Gracias por la ayuda.
>> This is not happening.	Esto no está pasando.
>> To our surprise, Tom came to our party with Mary.	Para nuestra sorpresa, Tom vino con Mary a nuestra fiesta.


In [10]:
# Q. 전처리 함수를 만들어 보세요. 아래 기능을 추가해주세요.
def preprocess_sentence(sentence):
    sentence = sentence.lower() # 대문자를 소문자로 변환
    sentence = re.sub(r' {2,}', ' ', sentence) # 둘 이상의 공백을 하나의 공백으로 치환
    sentence = sentence.strip() # 문자열 양 끝 공백 제거
    return sentence

In [11]:
spa_eng_sentences = list(map(preprocess_sentence, spa_eng_sentences))

print('슝=3')

슝=3


In [12]:
test_sentence_count = total_sentence_count // 200
print("Test Size: ", test_sentence_count)
print("\n")

train_spa_eng_sentences = spa_eng_sentences[:-test_sentence_count]
test_spa_eng_sentences = spa_eng_sentences[-test_sentence_count:]
print("Train Example:", len(train_spa_eng_sentences))
for sen in train_spa_eng_sentences[0:100][::20]:
    print(">>", sen)
print("\n")
print("Test Example:", len(test_spa_eng_sentences))
for sen in test_spa_eng_sentences[0:100][::20]:
    print(">>", sen)

Test Size:  594


Train Example: 118370
>> you're drunk.	estás borracha.
>> tom doesn't care about what mary says about him.	a tom no le importa lo que mary diga acerca de él.
>> thank you for the help.	gracias por la ayuda.
>> this is not happening.	esto no está pasando.
>> to our surprise, tom came to our party with mary.	para nuestra sorpresa, tom vino con mary a nuestra fiesta.


Test Example: 594
>> yesterday i finished the job.	ayer terminé el trabajo.
>> oslo is the capital of norway.	oslo es la capital de noruega.
>> you'll soon get used to the climate here.	pronto te vas a acostumbrar al clima de aquí.
>> i know he is watching me.	yo sé que él me está observando.
>> why doesn't tom want to go?	¿por qué tom no quiere ir?


In [13]:
def split_spa_eng_sentences(spa_eng_sentences):
    spa_sentences = []
    eng_sentences = []
    for spa_eng_sentence in tqdm(spa_eng_sentences):
        eng_sentence, spa_sentence = spa_eng_sentence.split('\t')
        spa_sentences.append(spa_sentence)
        eng_sentences.append(eng_sentence)
    return eng_sentences, spa_sentences

print('슝=3')

슝=3


In [14]:
train_eng_sentences, train_spa_sentences = split_spa_eng_sentences(train_spa_eng_sentences)
print(len(train_eng_sentences))
print(train_eng_sentences[0])
print('\n')
print(len(train_spa_sentences))
print(train_spa_sentences[0])

  0%|          | 0/118370 [00:00<?, ?it/s]

118370
you're drunk.


118370
estás borracha.


In [15]:
test_eng_sentences, test_spa_sentences = split_spa_eng_sentences(test_spa_eng_sentences)
print(len(test_eng_sentences))
print(test_eng_sentences[0])
print('\n')
print(len(test_spa_sentences))
print(test_spa_sentences[0])

  0%|          | 0/594 [00:00<?, ?it/s]

594
yesterday i finished the job.


594
ayer terminé el trabajo.


In [16]:
def generate_tokenizer(corpus,
                       vocab_size,
                       lang="spa-eng",
                       pad_id=0,   # pad token의 일련번호
                       bos_id=1,  # 문장의 시작을 의미하는 bos token(<s>)의 일련번호
                       eos_id=2,  # 문장의 끝을 의미하는 eos token(</s>)의 일련번호
                       unk_id=3):   # unk token의 일련번호
    file = "./%s_corpus.txt" % lang
    model = "%s_spm" % lang

    with open(file, 'w') as f:
        for row in corpus: f.write(str(row) + '\n')

    import sentencepiece as spm
    spm.SentencePieceTrainer.Train(
        '--input=./%s --model_prefix=%s --vocab_size=%d'\
        % (file, model, vocab_size) + \
        '--pad_id=%d --bos_id=%d --eos_id=%d --unk_id=%d'\
        % (pad_id, bos_id, eos_id, unk_id)
    )

    tokenizer = spm.SentencePieceProcessor()
    tokenizer.Load('%s.model' % model)

    return tokenizer

print("슝=3")

슝=3


In [17]:
VOCAB_SIZE = 20000
tokenizer = generate_tokenizer(train_eng_sentences + train_spa_sentences, VOCAB_SIZE, 'spa-eng')
tokenizer.set_encode_extra_options("bos:eos")  # 문장 양 끝에 <s> , </s> 추가

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=././spa-eng_corpus.txt --model_prefix=spa-eng_spm --vocab_size=20000--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ././spa-eng_corpus.txt
  input_format: 
  model_prefix: spa-eng_spm
  model_type: UNIGRAM
  vocab_size: 20000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all

True

In [18]:
def make_corpus(sentences, tokenizer):
    corpus = []
    for sentence in tqdm(sentences):
        tokens = tokenizer.encode_as_ids(sentence)
        corpus.append(tokens)
    return corpus

print('슝=3')

슝=3


In [19]:
eng_corpus = make_corpus(train_eng_sentences, tokenizer)
spa_corpus = make_corpus(train_spa_sentences, tokenizer)

  0%|          | 0/118370 [00:00<?, ?it/s]

  0%|          | 0/118370 [00:00<?, ?it/s]

In [20]:
print(train_eng_sentences[0])
print(eng_corpus[0])
print('\n')
print(train_spa_sentences[0])
print(spa_corpus[0])

you're drunk.
[1, 12, 7, 81, 1180, 0, 2]


estás borracha.
[1, 298, 16899, 0, 2]


In [21]:
MAX_LEN = 50

def pad_sequences_custom(sequences, max_len=50, pad_value=0):
    """
    sequences: list of list (각 문장별 토큰 ID 리스트)
    max_len: 고정할 최대 시퀀스 길이
    pad_value: 패딩에 사용할 값 (일반적으로 0)
    """
    padded_sequences = []

    for seq in sequences:
        # 초과 길이는 자르고
        if len(seq) > max_len:
            seq = seq[:max_len]
        # 부족한 길이는 pad_value로 채우기
        else:
            seq = seq + [pad_value] * (max_len - len(seq))

        padded_sequences.append(seq)

    # 최종적으로 torch.Tensor로 변환 (shape: [batch_size, max_len])
    return torch.tensor(padded_sequences, dtype=torch.long)

enc_ndarray = pad_sequences_custom(eng_corpus, max_len=MAX_LEN, pad_value=0)
dec_ndarray = pad_sequences_custom(spa_corpus, max_len=MAX_LEN, pad_value=0)

print(enc_ndarray.shape)  # 예) [batch_size, 50]
print(dec_ndarray.shape)  # 예) [batch_size, 50]
print("슝=3")

torch.Size([118370, 50])
torch.Size([118370, 50])
슝=3


In [22]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = TensorDataset(enc_ndarray, dec_ndarray)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

print("슝=3")

슝=3


In [23]:
# Positional Encoding 구현
def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, (2*(i//2)) / np.float32(d_model))

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table
print("슝=3")

슝=3


In [24]:
def generate_padding_mask(seq: torch.Tensor) -> torch.Tensor:
    """
    seq: shape [batch_size, seq_len]의 입력 (토큰 ID 텐서)
    반환: shape [batch_size, 1, 1, seq_len]의 패딩 마스크
         (seq == 0)인 위치가 1, 나머지는 0
    """
    # (seq == 0)은 불리언 텐서를 반환 -> float()로 형변환 -> (1.0 or 0.0)
    # 차원 확장: [batch_size, seq_len] → [batch_size, 1, 1, seq_len]
    return (seq == 0).unsqueeze(1).unsqueeze(2).float()


def generate_lookahead_mask(size: int) -> torch.Tensor:
    """
    size: 문장(시퀀스) 길이
    반환: shape [size, size],
         i < j (대각선 위)에 해당하는 위치가 1, 아닌 곳은 0
         (미래 토큰을 가리기 위한 마스크)
    """
    # triu(diagonal=1)은 주대각선 위가 1, 아래가 0인 텐서를 만들어 줌
    return torch.triu(torch.ones(size, size), diagonal=1)


def generate_masks(src: torch.Tensor, tgt: torch.Tensor):
    """
    src, tgt: shape [batch_size, seq_len]
    3가지 마스크를 반환:
      - enc_mask: 인코더 입력용 패딩 마스크
      - dec_enc_mask: 디코더-인코더 어텐션용 패딩 마스크
      - dec_mask: 디코더 자기어텐션용 마스크(룩어헤드 + 패딩)

    각각의 shape:
      - enc_mask, dec_enc_mask: [batch_size, 1, 1, src_seq_len]
      - dec_mask: [batch_size, 1, tgt_seq_len, tgt_seq_len]
    """
    # 1) 인코더 입력용 패딩 마스크
    enc_mask = generate_padding_mask(src)
    # 2) 디코더에서 인코더 값을 볼 때 사용하는 마스크 (src 마스크 재사용)
    dec_enc_mask = generate_padding_mask(src)

    # 3) 디코더 자기어텐션 마스크 (미래 토큰 방지 룩어헤드 + tgt 자체 패딩 마스크)
    dec_lookahead_mask = generate_lookahead_mask(tgt.shape[1])  # [tgt_seq_len, tgt_seq_len]
    dec_tgt_padding_mask = generate_padding_mask(tgt)           # [batch_size, 1, 1, tgt_seq_len]

    # 룩어헤드 마스크를 (batch 차원과 head 차원을 가상으로) 확장
    dec_lookahead_mask = dec_lookahead_mask.unsqueeze(0).unsqueeze(1)  # [1, 1, seq_len, seq_len]

    # 패딩 + 룩어헤드 마스크 병합
    # 브로드캐스팅에 의해 shape [batch_size, 1, tgt_seq_len, tgt_seq_len]이 됨

    dec_tgt_padding_mask = dec_tgt_padding_mask.to(device)
    dec_lookahead_mask = dec_lookahead_mask.to(device)

    dec_mask = torch.max(dec_tgt_padding_mask, dec_lookahead_mask)

    return enc_mask, dec_enc_mask, dec_mask

print("슝=3")

슝=3


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model을 num_heads로 나눈 만큼이 각 head가 담당할 차원 수
        self.depth = d_model // num_heads

        # Query, Key, Value를 구하는 선형 레이어
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 최종적으로 head들의 출력을 결합해주는 선형 레이어
        self.linear = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q, K, V:  [batch_size, num_heads, seq_len, depth]
        mask:     [batch_size, 1, seq_len, seq_len] 혹은
                  [batch_size, num_heads, seq_len, seq_len]
                  (어텐션에서 제외할 위치=1, 사용할 위치=0)
        """
        # d_k = depth
        d_k = Q.size(-1)  # K.shape[-1]도 동일
        # Q와 K의 전치 곱: (batch_size, num_heads, seq_len, seq_len)
        QK = torch.matmul(Q, K.transpose(-1, -2))

        # 스케일링
        scaled_qk = QK / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 마스크가 있는 경우 -1e9(매우 작은 수)를 더하여 softmax 후 확률이 0에 가깝도록 처리
        if mask is not None:
            scaled_qk = scaled_qk + (mask * -1e9)

        attentions = F.softmax(scaled_qk, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)
        out = torch.matmul(attentions, V)         # (batch_size, num_heads, seq_len, depth)

        return out, attentions

    def split_heads(self, x):
        """
        x: [batch_size, seq_len, d_model]
        반환: [batch_size, num_heads, seq_len, depth]
        """
        bsz, seq_len, _ = x.size()
        # d_model -> (num_heads * depth)이므로 view로 재배치
        x = x.view(bsz, seq_len, self.num_heads, self.depth)
        # (batch_size, seq_len, num_heads, depth) -> (batch_size, num_heads, seq_len, depth)
        x = x.permute(0, 2, 1, 3)
        return x

    def combine_heads(self, x):
        """
        x: [batch_size, num_heads, seq_len, depth]
        반환: [batch_size, seq_len, d_model]
        """
        bsz, num_heads, seq_len, depth = x.size()
        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, seq_len, self.d_model)
        return x

    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch_size, seq_len, d_model]
        mask:    [batch_size, 1, seq_len, seq_len] 혹은
                 [batch_size, num_heads, seq_len, seq_len]
        """
        # W_q, W_k, W_v는 각각 (d_model -> d_model) 선형 변환
        WQ = self.W_q(Q)  # [batch_size, seq_len, d_model]
        WK = self.W_k(K)  # [batch_size, seq_len, d_model]
        WV = self.W_v(V)  # [batch_size, seq_len, d_model]

        # 멀티헤드 분할
        WQ_splits = self.split_heads(WQ)  # [batch_size, num_heads, seq_len, depth]
        WK_splits = self.split_heads(WK)
        WV_splits = self.split_heads(WV)

        # Scaled dot-product attention
        out, attention_weights = self.scaled_dot_product_attention(
            WQ_splits, WK_splits, WV_splits, mask
        )

        # head 결과 결합 후 최종 선형
        out = self.combine_heads(out)  # [batch_size, seq_len, d_model]
        out = self.linear(out)         # [batch_size, seq_len, d_model]

        return out, attention_weights

print("슝=3")

슝=3


In [26]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.fc1(x))  # 첫 번째 Dense + ReLU
        out = self.fc2(out)          # 두 번째 Dense
        return out

print("슝=3")

슝=3


In [27]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        # nn.LayerNorm은 마지막 차원(d_model)을 기준으로 정규화
        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Multi-Head Attention 단계
        residual = x
        out = self.norm_1(x)
        out, enc_attn = self.enc_self_attn(out, out, out, mask)
        out = self.do(out)
        out = out + residual  # residual connection

        # Position-Wise Feed Forward 단계
        residual = out
        out = self.norm_2(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual  # residual connection

        return out, enc_attn

print("슝=3")

슝=3


In [28]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.dec_self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PoswiseFeedForwardNet(d_model, d_ff)

        self.norm_1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_2 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm_3 = nn.LayerNorm(d_model, eps=1e-6)

        self.do = nn.Dropout(dropout)

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        # Masked Multi-Head Attention
        residual = x
        out = self.norm_1(x)
        out, dec_attn = self.dec_self_attn(out, out, out, mask=padding_mask)
        out = self.do(out)
        out = out + residual

        # Encoder-Decoder Multi-Head Attention (주의: Q, K, V 순서)
        residual = out
        out = self.norm_2(out)
        out, dec_enc_attn = self.enc_dec_attn(out, enc_out, enc_out, mask=dec_enc_mask)
        out = self.do(out)
        out = out + residual

        # Position-Wise Feed Forward Network
        residual = out
        out = self.norm_3(out)
        out = self.ffn(out)
        out = self.do(out)
        out = out + residual

        return out, dec_attn, dec_enc_attn

print("슝=3")

슝=3


In [29]:
class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.n_layers = n_layers
        self.enc_layers = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.do = nn.Dropout(dropout)  # 필요 시 입력에 dropout 적용 가능

    def forward(self, x, mask):
        out = x
        enc_attns = []
        for i in range(self.n_layers):
            out, enc_attn = self.enc_layers[i](out, mask)
            enc_attns.append(enc_attn)
        return out, enc_attns

# 사용 예시: Encoder 인스턴스 생성 후 forward 호출
# encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
# out, enc_attns = encoder(x, mask)
print("슝=3")

슝=3


In [30]:
class Decoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout):
        super(Decoder, self).__init__()
        self.n_layers = n_layers
        self.dec_layers = nn.ModuleList(
            [DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

    def forward(self, x, enc_out, dec_enc_mask, padding_mask):
        out = x
        dec_attns = []
        dec_enc_attns = []
        for i in range(self.n_layers):
            out, dec_attn, dec_enc_attn = self.dec_layers[i](out, enc_out, dec_enc_mask, padding_mask)
            dec_attns.append(dec_attn)
            dec_enc_attns.append(dec_enc_attn)
        return out, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [31]:
import math

class Transformer(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff,
                 src_vocab_size, tgt_vocab_size, pos_len,
                 dropout=0.2, shared_fc=True, shared_emb=False):
        super(Transformer, self).__init__()
        # d_model은 스케일링에 사용되므로 float으로 저장
        self.d_model = float(d_model)

        # Embedding 레이어: shared_emb True면 동일한 임베딩을 사용합니다.
        if shared_emb:
            self.enc_emb = self.dec_emb = nn.Embedding(src_vocab_size, d_model)
        else:
            self.enc_emb = nn.Embedding(src_vocab_size, d_model)
            self.dec_emb = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding (넘파이 버전 결과를 torch.Tensor로 변환)
        pos_encoding_np = positional_encoding(pos_len, d_model)
        # 파라미터로 등록하지 않고 고정값이므로 buffer로 등록합니다.
        self.register_buffer("pos_encoding", torch.tensor(pos_encoding_np, dtype=torch.float32))

        self.do = nn.Dropout(dropout)

        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        self.fc = nn.Linear(d_model, tgt_vocab_size)

        self.shared_fc = shared_fc
        if shared_fc:
            # fc 레이어와 디코더 임베딩의 weight를 공유합니다.
            self.fc.weight = self.dec_emb.weight

    def embedding(self, emb, x):
        """
        emb: 임베딩 레이어
        x: [batch_size, seq_len] (토큰 인덱스)
        """
        seq_len = x.size(1)
        out = emb(x)  # [batch_size, seq_len, d_model]
        if self.shared_fc:
            out = out * math.sqrt(self.d_model)
        # pos_encoding: [pos_len, d_model] → [1, pos_len, d_model] 후 슬라이싱
        out = out + self.pos_encoding[:seq_len, :].unsqueeze(0)
        out = self.do(out)
        return out

    def forward(self, enc_in, dec_in, enc_mask, dec_enc_mask, dec_mask):
        """
        enc_in: [batch_size, src_seq_len]
        dec_in: [batch_size, tgt_seq_len]
        enc_mask, dec_enc_mask, dec_mask: 마스킹 텐서들
        """
        # Embedding 및 positional encoding 적용
        enc_in_emb = self.embedding(self.enc_emb, enc_in)
        dec_in_emb = self.embedding(self.dec_emb, dec_in)

        # Encoder와 Decoder 통과
        enc_out, enc_attns = self.encoder(enc_in_emb, enc_mask)
        dec_out, dec_attns, dec_enc_attns = self.decoder(dec_in_emb, enc_out, dec_enc_mask, dec_mask)

        logits = self.fc(dec_out)
        return logits, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


In [32]:
# 주어진 하이퍼파라미터로 Transformer 인스턴스 생성
transformer = Transformer(
    n_layers=2,
    d_model=512,
    n_heads=8,
    d_ff=2048,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200,
    dropout=0.3,
    shared_fc=True,
    shared_emb=True)

transformer = transformer.to(device)

d_model = 512

print("슝=3")

슝=3


In [33]:
class LearningRateScheduler:
    def __init__(self, d_model, warmup_steps=60): # 4000
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        # step을 float으로 변환하여 지수 연산이 제대로 수행되도록 함
        step = float(step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

print("슝=3")

슝=3


In [34]:
# Learning Rate 인스턴스 선언
learning_rate = LearningRateScheduler(d_model)

# 초기 lr은 스텝 1에 해당하는 값으로 설정합니다.
optimizer = torch.optim.Adam(transformer.parameters(),
                             lr=learning_rate(1),
                             betas=(0.9, 0.98),
                             eps=1e-9)

print("슝=3")

슝=3


In [35]:
def loss_function(real, pred):
    """
    real: [batch_size, seq_len] (정답 토큰 인덱스)
    pred: [batch_size, seq_len, num_classes] (모델의 raw logits)
    """

    real = real.to(device)
    pred = pred.to(device)

    # 예측 값을 (N, C) 형태로 flatten하고, 정답도 flatten하여 개별 손실 값을 구함
    loss_ = F.cross_entropy(pred.contiguous().view(-1, pred.size(-1)), real.contiguous().view(-1), reduction='none')
    # 다시 (batch_size, seq_len)로 reshape
    loss_ = loss_.view(real.size())

    # real이 0이 아닌 위치에 대한 마스크 생성 (0이면 패딩 토큰)
    mask = (real != 0).float()
    loss_ = loss_ * mask

    # 전체 손실 합을 마스크 합으로 나누어 평균 손실 계산
    return loss_.sum() / mask.sum()

print("슝=3")

슝=3


In [36]:
def train_step(src, tgt, model, optimizer):
    model.train()  # 모델을 training 모드로 전환
    optimizer.zero_grad()

    # tgt의 오른쪽 시프트: decoder input과 gold target 분리
    tgt_in = tgt[:, :-1]  # Decoder의 입력
    gold = tgt[:, 1:]     # Decoder의 정답(target)

    # 마스크 생성 (generate_masks는 PyTorch용으로 변환된 함수여야 합니다)
    enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_in)

    src = src.to(device)
    tgt_in = tgt_in.to(device)
    enc_mask = enc_mask.to(device)
    dec_enc_mask = dec_enc_mask.to(device)
    dec_mask = dec_mask.to(device)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(src, tgt_in, enc_mask, dec_enc_mask, dec_mask)

    # loss 계산
    loss = loss_function(gold, predictions)

    # 역전파 수행 및 파라미터 업데이트
    loss.backward()
    optimizer.step()

    return loss, enc_attns, dec_attns, dec_enc_attns

print("슝=3")

슝=3


%%time

EPOCHS = 3

for epoch in range(EPOCHS):
    total_loss = 0.0
    dataset_count = len(train_dataloader)  # train_loader는 PyTorch DataLoader입니다.
    tqdm_bar = tqdm(total=dataset_count)

    for batch, (src, tgt) in enumerate(train_dataloader):
        # train_step 함수는 (loss, enc_attns, dec_attns, dec_enc_attns)를 반환합니다.
        loss, enc_attns, dec_attns, dec_enc_attns = train_step(src, tgt, transformer, optimizer)

        total_loss += loss.item()  # PyTorch에서는 loss.numpy() 대신 loss.item() 사용
        tqdm_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
        tqdm_bar.update(1)

    tqdm_bar.close()
    print(f"Epoch {epoch+1}, Loss: {total_loss / dataset_count:.4f}")

In [37]:
# 아래 두 문장을 바꿔가며 테스트 해보세요
reference = "많 은 자연어 처리 연구자 들 이 트랜스포머 를 선호 한다".split()
candidate = "적 은 자연어 학 개발자 들 가 트랜스포머 을 선호 한다 요".split()

print("원문:", reference)
print("번역문:", candidate)
print("BLEU Score:", sentence_bleu([reference], candidate))

원문: ['많', '은', '자연어', '처리', '연구자', '들', '이', '트랜스포머', '를', '선호', '한다']
번역문: ['적', '은', '자연어', '학', '개발자', '들', '가', '트랜스포머', '을', '선호', '한다', '요']
BLEU Score: 8.190757052088229e-155


/opt/conda/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/opt/conda/lib/python3.12/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [38]:
print("1-gram:", sentence_bleu([reference], candidate, weights=[1, 0, 0, 0]))
print("2-gram:", sentence_bleu([reference], candidate, weights=[0, 1, 0, 0]))
print("3-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 1, 0]))
print("4-gram:", sentence_bleu([reference], candidate, weights=[0, 0, 0, 1]))

1-gram: 0.5
2-gram: 0.18181818181818182
3-gram: 2.2250738585072626e-308
4-gram: 2.2250738585072626e-308


In [39]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                         candidate,
                         weights=weights,
                         smoothing_function=SmoothingFunction().method1)  # smoothing_function 적용

print("BLEU-1:", calculate_bleu(reference, candidate, weights=[1, 0, 0, 0]))
print("BLEU-2:", calculate_bleu(reference, candidate, weights=[0, 1, 0, 0]))
print("BLEU-3:", calculate_bleu(reference, candidate, weights=[0, 0, 1, 0]))
print("BLEU-4:", calculate_bleu(reference, candidate, weights=[0, 0, 0, 1]))

print("\nBLEU-Total:", calculate_bleu(reference, candidate))

BLEU-1: 0.5
BLEU-2: 0.18181818181818182
BLEU-3: 0.010000000000000004
BLEU-4: 0.011111111111111112

BLEU-Total: 0.05637560315259291


In [40]:
import torch
import torch.nn.functional as F

def translate(tokens, model, src_tokenizer, tgt_tokenizer):
    # tokens: 입력 토큰 리스트
    # MAX_LEN: 최대 길이 (전역 변수 혹은 상수)
    # device: 모델과 데이터가 위치한 디바이스

    # tokens 길이가 MAX_LEN보다 크면 자르고, 작으면 0으로 패딩
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
    else:
        tokens = tokens + [0] * (MAX_LEN - len(tokens))

    # 배치 차원을 추가하여 텐서로 변환 (shape: [1, MAX_LEN])
    padded_tokens = torch.tensor([tokens], dtype=torch.long, device=device)

    ids = []
    # 디코더의 첫 입력은 BOS 토큰 (배치 차원 추가)
    output = torch.tensor([[tgt_tokenizer.bos_id()]], dtype=torch.long, device=device)

    for i in range(MAX_LEN):
        # generate_masks는 padded_tokens와 현재 output으로부터 마스크들을 생성합니다.
        enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(padded_tokens, output)

        # 모델 예측: predictions shape: [batch, seq_len, num_classes]
        predictions, _, _, _ = model(padded_tokens, output, enc_padding_mask, combined_mask, dec_padding_mask)

        # 마지막 시퀀스 위치의 예측값을 소프트맥스 후 argmax로 선택
        predicted_id = predictions[0, -1].softmax(dim=-1).argmax(dim=-1).item()

        # EOS 토큰에 도달하면 현재까지의 예측 토큰 ids를 디코딩 후 반환
        if tgt_tokenizer.eos_id() == predicted_id:
            result = tgt_tokenizer.decode_ids(ids)
            return result

        ids.append(predicted_id)
        # 현재 output에 새로운 예측 토큰을 연결 (dim=1)
        new_token = torch.tensor([[predicted_id]], dtype=torch.long, device=device)
        output = torch.cat([output, new_token], dim=1)

    result = tgt_tokenizer.decode_ids(ids)
    return result

print("슝=3")

슝=3


In [41]:
def eval_bleu_single(model, src_sentence, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    src_tokens = src_tokenizer.encode_as_ids(src_sentence)
    tgt_tokens = tgt_tokenizer.encode_as_ids(tgt_sentence)

    if (len(src_tokens) > MAX_LEN): return None
    if (len(tgt_tokens) > MAX_LEN): return None

    reference = tgt_sentence.split()
    candidate = translate(src_tokens, model, src_tokenizer, tgt_tokenizer).split()

    score = sentence_bleu([reference], candidate,
                          smoothing_function=SmoothingFunction().method1)

    if verbose:
        print("Source Sentence: ", src_sentence)
        print("Model Prediction: ", candidate)
        print("Real: ", reference)
        print("Score: %lf\n" % score)

    return score

print('슝=3')

슝=3


In [42]:
# Q. 인덱스를 바꿔가며 테스트해 보세요
test_idx = 0

eval_bleu_single(transformer,
                 test_eng_sentences[test_idx],
                 test_spa_sentences[test_idx],
                 tokenizer,
                 tokenizer)

Source Sentence:  yesterday i finished the job.
Model Prediction:  []
Real:  ['ayer', 'terminé', 'el', 'trabajo.']
Score: 0.000000



0

In [43]:
def eval_bleu(model, src_sentences, tgt_sentence, src_tokenizer, tgt_tokenizer, verbose=True):
    total_score = 0.0
    sample_size = len(src_sentences)

    for idx in tqdm(range(sample_size)):
        score = eval_bleu_single(model, src_sentences[idx], tgt_sentence[idx], src_tokenizer, tgt_tokenizer, verbose)
        if not score: continue

        total_score += score

    print("Num of Sample:", sample_size)
    print("Total Score:", total_score / sample_size)

print("슝=3")

슝=3


In [44]:
eval_bleu(transformer, test_eng_sentences, test_spa_sentences, tokenizer, tokenizer, verbose=False)

  0%|          | 0/594 [00:00<?, ?it/s]

Num of Sample: 594
Total Score: 0.0


In [45]:
def beam_search_decoder(prob, beam_size):
    sequences = [[[], 1.0]]  # 생성된 문장과 점수를 저장

    for tok in prob:
        all_candidates = []

        for seq, score in sequences:
            for idx, p in enumerate(tok): # 각 단어의 확률을 총점에 누적 곱
                candidate = [seq + [idx], score * -math.log(-(p-1))]
                all_candidates.append(candidate)

        ordered = sorted(all_candidates,
                         key=lambda tup:tup[1],
                         reverse=True) # 총점 순 정렬
        sequences = ordered[:beam_size] # Beam Size에 해당하는 문장만 저장

    return sequences

print("슝=3")

슝=3


In [46]:
vocab = {
    0: "<pad>",
    1: "까요?",
    2: "커피",
    3: "마셔",
    4: "가져",
    5: "될",
    6: "를",
    7: "한",
    8: "잔",
    9: "도",
}

prob_seq = [[0.01, 0.01, 0.60, 0.32, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.75, 0.01, 0.01, 0.17],
            [0.01, 0.01, 0.01, 0.35, 0.48, 0.10, 0.01, 0.01, 0.01, 0.01],
            [0.24, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.68],
            [0.01, 0.01, 0.12, 0.01, 0.01, 0.80, 0.01, 0.01, 0.01, 0.01],
            [0.01, 0.81, 0.01, 0.01, 0.01, 0.01, 0.11, 0.01, 0.01, 0.01],
            [0.70, 0.22, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
            [0.91, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]]

prob_seq = np.array(prob_seq)
beam_size = 3

result = beam_search_decoder(prob_seq, beam_size)

for seq, score in result:
    sentence = ""

    for word in seq:
        sentence += vocab[word] + " "

    print(sentence, "// Score: %.4f" % score)

커피 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 42.5243
커피 를 마셔 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 28.0135
마셔 를 가져 도 될 까요? <pad> <pad> <pad> <pad>  // Score: 17.8983


In [47]:
import torch
import torch.nn.functional as F

def calc_prob(src_ids, tgt_ids, model):
    # 마스크 생성 (PyTorch 버전)
    enc_padding_mask, combined_mask, dec_padding_mask = generate_masks(src_ids, tgt_ids)

    # 모델 forward pass
    predictions, enc_attns, dec_attns, dec_enc_attns = model(
        src_ids,
        tgt_ids,
        enc_padding_mask,
        combined_mask,
        dec_padding_mask
    )

    # 마지막 차원에 대해 softmax 적용하여 확률값 계산
    return F.softmax(predictions, dim=-1)

print("슝=3")

슝=3


In [48]:
import numpy as np
import torch

def beam_search_decoder(sentence,
                        src_len,
                        tgt_len,
                        model,
                        src_tokenizer,
                        tgt_tokenizer,
                        beam_size):
    # 입력 문장을 토큰화
    tokens = src_tokenizer.encode_as_ids(sentence)

    # src_in: [1, src_len] 크기의 텐서로 padding (0: 패딩 토큰)
    padded = np.zeros((1, src_len), dtype=np.int64)
    padded[0, :len(tokens)] = tokens
    src_in = torch.tensor(padded, dtype=torch.long, device=device)

    # beam search용 캐시 배열들
    pred_cache = np.zeros((beam_size * beam_size, tgt_len), dtype=np.int64)
    pred_tmp = np.zeros((beam_size, tgt_len), dtype=np.int64)

    eos_flag = np.zeros((beam_size,), dtype=np.int64)  # EOS를 만난 branch 표시 (EOS: -1)
    scores = np.ones((beam_size,), dtype=np.float32)     # 각 branch의 score (확률 곱)

    # 디코더 첫 입력은 BOS 토큰
    pred_tmp[:, 0] = tgt_tokenizer.bos_id()

    # 초기 디코더 입력 (branch 0의 첫 토큰) -> shape: [1, 1]
    dec_in = torch.tensor(pred_tmp[0, :1], dtype=torch.long, device=device).unsqueeze(0)
    # calc_prob()는 softmax를 적용한 확률 텐서를 반환함
    prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

    # seq_pos: 디코더 시퀀스 위치
    for seq_pos in range(1, tgt_len):
        score_cache = np.ones((beam_size * beam_size,), dtype=np.float32)

        # 각 beam branch에 대해 캐시 초기화
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            score_cache[cache_pos:cache_pos+beam_size] = scores[branch_idx]
            pred_cache[cache_pos:cache_pos+beam_size, :seq_pos] = pred_tmp[branch_idx, :seq_pos]

        # 각 beam branch에 대해 후보 확률 계산 및 캐시 업데이트
        for branch_idx in range(beam_size):
            cache_pos = branch_idx * beam_size
            if seq_pos != 1:
                # 해당 branch의 현재까지의 시퀀스를 디코더 입력으로 변환
                dec_in_np = pred_cache[branch_idx, :seq_pos]
                dec_in = torch.tensor(dec_in_np, dtype=torch.long, device=device).unsqueeze(0)
                prob = calc_prob(src_in, dec_in, model)[0, -1].detach().cpu().numpy()

            # 각 branch 내에서 beam_size만큼의 후보 토큰을 선택
            for beam_idx in range(beam_size):
                max_idx = np.argmax(prob)
                # 후보 branch의 score 업데이트 (곱셈으로 누적)
                score_cache[cache_pos + beam_idx] *= prob[max_idx]
                pred_cache[cache_pos + beam_idx, seq_pos] = max_idx
                # 이미 선택된 토큰은 다시 선택되지 않도록 -1로 마킹
                prob[max_idx] = -1

        # 각 beam branch에서 최고 score를 가진 후보를 선택
        for beam_idx in range(beam_size):
            if eos_flag[beam_idx] == -1:
                continue
            max_idx = np.argmax(score_cache)
            prediction = pred_cache[max_idx, :seq_pos+1].copy()
            pred_tmp[beam_idx, :seq_pos+1] = prediction
            scores[beam_idx] = score_cache[max_idx]
            score_cache[max_idx] = -1  # 해당 후보 제거

            # 만약 EOS 토큰이면 해당 branch는 종료 표시 (-1)
            if prediction[-1] == tgt_tokenizer.eos_id():
                eos_flag[beam_idx] = -1

    # 각 branch의 예측 시퀀스에서 EOS 토큰 이전까지만 추출하여 결과 반환
    pred = []
    for long_pred in pred_tmp:
        eos_token = tgt_tokenizer.eos_id()
        # EOS 토큰이 없는 경우, 전체 시퀀스를 사용하도록 처리할 수 있음
        try:
            eos_idx = list(long_pred).index(eos_token)
        except ValueError:
            eos_idx = tgt_len - 1
        short_pred = long_pred[:eos_idx+1]
        pred.append(short_pred.tolist())

    return pred

print("슝=3")

슝=3


In [49]:
def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    return sentence_bleu([reference],
                            candidate,
                            weights=weights,
                            smoothing_function=SmoothingFunction().method1)

print('슝=3')

슝=3


In [50]:
def beam_bleu(reference, ids, tokenizer):
    # 기준 문장을 토큰화
    reference_tokens = reference.split()

    total_score = 0.0
    num_candidates = len(ids)
    if num_candidates == 0:
        return 0.0

    for candidate_ids in ids:
        # 후보 문장을 디코딩 후 토큰화
        candidate_sentence = tokenizer.decode_ids(candidate_ids)
        candidate_tokens = candidate_sentence.split()

        score = calculate_bleu(reference_tokens, candidate_tokens)

        print(f"Reference: {reference_tokens}")
        print(f"Candidate: {candidate_tokens}")
        print(f"BLEU: {score}")

        total_score += score

    return total_score / num_candidates

print("슝=3")

슝=3


In [51]:
# Q. 인덱스를 바꿔가며 확인해 보세요
test_idx = 1

ids = \
beam_search_decoder(test_eng_sentences[test_idx],
                    MAX_LEN,
                    MAX_LEN,
                    transformer,
                    tokenizer,
                    tokenizer,
                    beam_size=5)

bleu = beam_bleu(test_spa_sentences[test_idx], ids, tokenizer)
print(bleu)

Reference: ['este', 'perro', 'es', 'blanco.']
Candidate: []
BLEU: 0
Reference: ['este', 'perro', 'es', 'blanco.']
Candidate: ['.']
BLEU: 0
Reference: ['este', 'perro', 'es', 'blanco.']
Candidate: []
BLEU: 0
Reference: ['este', 'perro', 'es', 'blanco.']
Candidate: []
BLEU: 0
Reference: ['este', 'perro', 'es', 'blanco.']
Candidate: []
BLEU: 0
0.0


In [52]:
!pip install gensim

In [53]:
import gensim.downloader as api

wv = api.load('glove-wiki-gigaword-300')

In [54]:
wv.most_similar("banana")

[('bananas', 0.6691170930862427),
 ('mango', 0.5804104208946228),
 ('pineapple', 0.5492372512817383),
 ('coconut', 0.5462778806686401),
 ('papaya', 0.541056752204895),
 ('fruit', 0.52181077003479),
 ('growers', 0.4877638816833496),
 ('nut', 0.48399588465690613),
 ('peanut', 0.48062023520469666),
 ('potato', 0.48061180114746094)]

In [55]:
sample_sentence = "you know ? all you need is attention ."
sample_tokens = sample_sentence.split()

selected_tok = random.choice(sample_tokens)

result = ""
for tok in sample_tokens:
    if tok is selected_tok:
        result += wv.most_similar(tok)[0][0] + " "

    else:
        result += tok + " "

print("From:", sample_sentence)
print("To:", result)

From: you know ? all you need is attention .
To: you know ? all you need is attention , 


In [56]:
# Q. Lexical Substitution 을 구현해봅시다.
def lexical_sub(sentence, wv):
    # 문장을 토큰화
    tokens = sentence.split()

    # 유효한 단어 필터링 (임베딩에 존재하는 단어만 고려)
    valid_tokens = [tok for tok in tokens if tok in wv]

    # 대체할 단어 선택 (임베딩 내 존재하는 단어 중 하나)
    if not valid_tokens:
        return sentence  # 모든 단어가 임베딩 내에 없으면 원래 문장 반환

    selected_tok = random.choice(valid_tokens)

    # 가장 유사한 단어 찾기
    similar_word = wv.most_similar(selected_tok)[0][0]

    # 변환된 문장 생성
    new_sentence = " ".join([similar_word if tok == selected_tok else tok for tok in tokens])

    return new_sentence

In [57]:
new_corpus = []

for old_src in tqdm(test_eng_sentences):
    new_src = lexical_sub(old_src, wv)
    if new_src is not None:
        new_corpus.append(new_src)
    # Augmentation이 없더라도 원본 문장을 포함시킵니다
    new_corpus.append(old_src)

print(new_corpus[:10])

  0%|          | 0/594 [00:00<?, ?it/s]

['yesterday i finishing the job.', 'yesterday i finished the job.', 'it dog is white.', 'this dog is white.', "i'm going home. i've had so of tom.", "i'm going home. i've had enough of tom.", 'tom and mary are your best friends.', 'tom and mary are my best friends.', 'there is a bookstore along from my house.', 'there is a bookstore across from my house.']


## Step1. 데이터 다운로드

In [58]:
# 1. wget을 이용해 깃허브의 Raw 파일을 다운로드합니다.
!wget https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv

# 2. 잘 다운로드 되었는지 확인해볼까요?
import pandas as pd

chatbot_data = pd.read_csv('ChatbotData.csv')
print(f"데이터 로드 완료! 총 {len(chatbot_data)}개의 문장 쌍을 찾았습니다.")

# 3. 상위 5개 데이터 확인
chatbot_data.head()

--2026-03-11 09:54:25--  https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 889842 (869K) [text/plain]
Saving to: ‘ChatbotData.csv.2’

ChatbotData.csv.2   100%[===================>] 868.99K  5.31MB/s    in 0.2s    

2026-03-11 09:54:26 (5.31 MB/s) - ‘ChatbotData.csv.2’ saved [889842/889842]

데이터 로드 완료! 총 11823개의 문장 쌍을 찾았습니다.


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [59]:
import pandas as pd

# 1. ChatbotData.csv 파일을 읽어옵니다.
# 파일이 현재 작업 디렉토리에 있다고 가정합니다.
data_path = 'ChatbotData.csv'
df = pd.read_csv(data_path)

# 2. 데이터의 구성을 살짝 살펴볼까요? 
# 보통 'Q'(질문), 'A'(답변), 'label' 컬럼으로 이루어져 있습니다.
print("--- 데이터 상위 5개 출력 ---")
print(df.head())

# 3. 이미지 가이드대로 questions와 answers 변수에 각각 저장합니다.
# 리스트 형태로 저장해야 나중에 토큰화하기 편합니다.
questions = df['Q'].tolist()
answers = df['A'].tolist()

# 4. 제대로 저장되었는지 확인!
print("\n--- 데이터 로드 완료 ---")
print(f"전체 문장 쌍 개수: {len(questions)}개")
print(f"첫 번째 질문: {questions[0]}")
print(f"첫 번째 답변: {answers[0]}")

print("\n슝=3")

--- 데이터 상위 5개 출력 ---
                 Q            A  label
0           12시 땡!   하루가 또 가네요.      0
1      1지망 학교 떨어졌어    위로해 드립니다.      0
2     3박4일 놀러가고 싶다  여행은 언제나 좋죠.      0
3  3박4일 정도 놀러가고 싶다  여행은 언제나 좋죠.      0
4          PPL 심하네   눈살이 찌푸려지죠.      0

--- 데이터 로드 완료 ---
전체 문장 쌍 개수: 11823개
첫 번째 질문: 12시 땡!
첫 번째 답변: 하루가 또 가네요.

슝=3


## Step 2. 데이터 정제

In [60]:
import re

def preprocess_sentence(sentence):
    # 1. 영문자 소문자 변환 및 양끝 공백 제거
    sentence = sentence.lower().strip()

    # 2. 영문자, 한글, 숫자, 주요 특수문자(?.!,)를 제외한 모든 것 제거
    sentence = re.sub(r"[^a-zA-Z가-힣0-9?.!, ]+", "", sentence)

    # 3. 여러 개의 공백을 하나의 공백으로 합치기 (수정된 부분!)
    # \s+ 는 모든 종류의 공백(스페이스, 탭 등)이 하나 이상 반복될 때를 의미합니다.
    sentence = re.sub(r"\s+", " ", sentence)
    
    # 4. 최종 양끝 공백 제거
    sentence = sentence.strip()
    
    return sentence

# --- 데이터 적용 테스트 ---
print("--- 정제 테스트 ---")
test_q = "Hi, 오늘 날씨 어때요? ^_^ 123!!"
print(f"원본: {test_q}")
print(f"결과: {preprocess_sentence(test_q)}")

# --- 전체 데이터에 적용 ---
# questions와 answers가 정의되어 있는지 확인하세요!
questions = [preprocess_sentence(q) for q in questions]
answers = [preprocess_sentence(a) for a in answers]

print("\n--- 전체 데이터 정제 완료 ---")
print(f"정제된 질문 1: {questions[0]}")
print(f"정제된 답변 1: {answers[0]}")

--- 정제 테스트 ---
원본: Hi, 오늘 날씨 어때요? ^_^ 123!!
결과: hi, 오늘 날씨 어때요? 123!!

--- 전체 데이터 정제 완료 ---
정제된 질문 1: 12시 땡!
정제된 답변 1: 하루가 또 가네요.


Step 3. 데이터 토큰화

In [61]:
# Mecab 설치 (Colab 환경일 경우 필요, 이미 설치되어 있다면 생략 가능)
# !git clone https://github.com/SOMWJH/konlpy-mecab-install.git
# !bash konlpy-mecab-install/install_mecab-ko_on_colab190912.sh

from konlpy.tag import Mecab
import tqdm

mecab = Mecab()

def build_corpus(src_data, tgt_data, tokenize_func, max_len=40):
    que_corpus = []
    ans_corpus = []
    
    # 중복을 체크하기 위한 셋(Set) - (질문, 답변) 쌍을 저장
    seen = set()

    for s, t in zip(src_data, tgt_data):
        # 1. 정제 (이미 밖에서 했다면 한 번 더 확인하는 차원)
        cleaned_s = preprocess_sentence(s)
        cleaned_t = preprocess_sentence(t)
        
        # 2. 토큰화
        s_tokens = tokenize_func(cleaned_s)
        t_tokens = tokenize_func(cleaned_t)
        
        # 3. 중복 제거 (쌍을 튜플로 묶어 set에 있는지 확인)
        # "쌍이 흐트러지지 않도록" 동일한 (질문, 답변) 쌍이 들어오는 것을 방지합니다.
        pair = (tuple(s_tokens), tuple(t_tokens))
        if pair in seen:
            continue
        
        # 4. 길이 제한 (토큰 개수가 max_len 이하인 경우만 저장)
        if len(s_tokens) <= max_len and len(t_tokens) <= max_len:
            que_corpus.append(s_tokens)
            ans_corpus.append(t_tokens)
            seen.add(pair) # 처리된 쌍은 저장

    return que_corpus, ans_corpus

# --- 실행 부분 ---
# 가이드대로 mecab.morphs 함수를 인자로 전달합니다.
que_corpus, ans_corpus = build_corpus(questions, answers, mecab.morphs)

print("--- 데이터 토큰화 완료 ---")
print(f"최종 말뭉치 쌍 개수: {len(que_corpus)}개")
print(f"질문 토큰화 예시: {que_corpus[0]}")
print(f"답변 토큰화 예시: {ans_corpus[0]}")

print("\n슝=3")

--- 데이터 토큰화 완료 ---
최종 말뭉치 쌍 개수: 11746개
질문 토큰화 예시: ['12', '시', '땡', '!']
답변 토큰화 예시: ['하루', '가', '또', '가', '네요', '.']

슝=3


Step 4. Augmentation

In [62]:
import zipfile
import os
import gensim
import random
from tqdm import tqdm

# 1. ko.zip 압축 해제
zip_file_path = 'ko.zip'
extract_path = '.' # 현재 폴더에 압축 해제

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ ko.zip 압축 해제 완료!")
else:
    print("❌ ko.zip 파일을 찾을 수 없습니다. 파일명을 확인해주세요.")

# 2. 한국어 Word2Vec 모델 로드 (ko.bin)
# 보통 Kyubyong님의 모델은 Word2Vec.load()를 사용합니다.
try:
    # 압축을 풀면 나오는 ko.bin 파일을 로드합니다.
    word2vec = gensim.models.Word2Vec.load('ko.bin')
    wv = word2vec.wv
    print("✅ 모델 로드 성공! 이제 단어의 의미를 파악할 수 있습니다.")
except Exception as e:
    print(f"❌ 모델 로드 중 에러 발생: {e}")

# 3. 어휘 치환(Lexical Substitution) 함수
def lexical_sub(tokens, wv):
    # 임베딩 사전에 있는 토큰들만 후보로 추출
    valid_tokens = [tok for tok in tokens if tok in wv]
    
    if not valid_tokens:
        return tokens # 바꿀 수 있는 단어가 없으면 원본 반환
    
    selected_tok = random.choice(valid_tokens)
    similar_word = wv.most_similar(selected_tok)[0][0]
    
    return [similar_word if tok == selected_tok else tok for tok in tokens]

# 4. 데이터 3배 증강 (Triple the Corpus)
# 원본(que_corpus, ans_corpus)이 리스트 형태로 메모리에 있어야 합니다.
aug_que_corpus = []
aug_ans_corpus = []

for q, a in tqdm(zip(que_corpus, ans_corpus), total=len(que_corpus), desc="3배 증강 중"):
    # Case 1: 원본 질문 + 원본 답변
    aug_que_corpus.append(q)
    aug_ans_corpus.append(a)
    
    # Case 2: 증강 질문 + 원본 답변
    aug_que_corpus.append(lexical_sub(q, wv))
    aug_ans_corpus.append(a)
    
    # Case 3: 원본 질문 + 증강 답변
    aug_que_corpus.append(q)
    aug_ans_corpus.append(lexical_sub(a, wv))

print("\n--- 증강 작업 종료 ---")
print(f"최종 데이터 개수: {len(aug_que_corpus)}개 (원본의 3배)")

✅ ko.zip 압축 해제 완료!
❌ 모델 로드 중 에러 발생: 'Word2Vec' object has no attribute 'wv'


3배 증강 중: 100%|██████████| 11746/11746 [08:25<00:00, 23.25it/s]


--- 증강 작업 종료 ---
최종 데이터 개수: 35238개 (원본의 3배)


Step 5. 데이터 벡터화

In [63]:
!pip install tensorflow

In [64]:
import tensorflow as tf

# 1. 타겟 데이터(aug_ans_corpus)에 <start>와 <end> 토큰 추가
# 리스트 더하기 기능을 이용해 모든 답변의 앞뒤에 토큰을 붙여줍니다.
aug_ans_corpus = [["<start>"] + ans + ["<end>"] for ans in aug_ans_corpus]

# 2. 통합 단어 사전 구축
# 질문(Source)과 답변(Target)을 합쳐서 하나의 토크나이저에 학습시킵니다.
all_corpus = aug_que_corpus + aug_ans_corpus

tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
tokenizer.fit_on_texts(all_corpus)

# 3. 정수 인코딩 (텍스트 -> 정수 ID)
enc_train = tokenizer.texts_to_sequences(aug_que_corpus)
dec_train = tokenizer.texts_to_sequences(aug_ans_corpus)

# 4. 패딩 (Sequence Padding)
# 모든 문장의 길이를 동일하게 맞춥니다. 'post'를 써서 뒤쪽에 0을 채웁니다.
enc_train = tf.keras.preprocessing.sequence.pad_sequences(enc_train, padding='post')
dec_train = tf.keras.preprocessing.sequence.pad_sequences(dec_train, padding='post')

print("--- 데이터 벡터화 및 패딩 완료 ---")
print(f"enc_train 크기: {enc_train.shape}")
print(f"dec_train 크기: {dec_train.shape}")
print(f"최종 단어 사전 크기 (Vocab Size): {len(tokenizer.word_index) + 1}") # 0번(패딩) 포함

# 샘플 확인
print("\n[인코딩 샘플]")
print(f"원본: {aug_que_corpus[0]}")
print(f"정수: {enc_train[0]}")

I0000 00:00:1773223375.441056    7293 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773223377.168873    7293 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


--- 데이터 벡터화 및 패딩 완료 ---
enc_train 크기: (35238, 32)
dec_train 크기: (35238, 42)
최종 단어 사전 크기 (Vocab Size): 6869

[인코딩 샘플]
원본: ['12', '시', '땡', '!']
정수: [2767  175 3331  116    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


Step 6. 훈련하기

In [65]:
#1. 하이퍼파라미터 및 모델 초기화
import torch
import torch.nn as nn
import torch.optim as optim


VOCAB_SIZE = len(vocab) # 이전 단계에서 만든 단어 사전 크기
N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2

# 모델 인스턴스 생성 (이미 정의된 Transformer 클래스가 있다고 가정)
# 만약 클래스명이 다르다면 본인의 클래스명으로 수정해주세요!
transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    pos_len=200, # 최대 문장 길이보다 넉넉하게
    dropout=DROPOUT
).to(device)

print("✅ 모델 준비 완료!")

✅ 모델 준비 완료!


In [66]:
#2. 학습률 스케줄러 및 손실 함수
class CustomSchedule:
    def __init__(self, d_model, warmup_steps=1000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
    
    def __call__(self, step):
        step = max(1, step)
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

# 손실 함수 (패딩 토큰 0은 학습에서 제외)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)

# 스케줄러 적용
lr_scheduler = CustomSchedule(D_MODEL)

In [67]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# =================================================================
# 1. 에러 원인 완벽 차단: 절대 터지지 않는 VOCAB_SIZE와 POS_LEN 계산
# =================================================================
# 데이터에 존재하는 가장 큰 숫자(ID)를 찾아 무조건 그것보다 1 크게 만듭니다.
ACTUAL_VOCAB_SIZE = int(np.max([enc_train.max(), dec_train.max()])) + 1

# 문장 길이도 데이터셋의 최대 길이보다 무조건 길게 설정합니다.
MAX_SEQ_LEN = max(enc_train.shape[1], dec_train.shape[1]) + 50 

print(f"📊 데이터 기준 안전한 VOCAB_SIZE: {ACTUAL_VOCAB_SIZE}")
print(f"📊 데이터 기준 안전한 POS_LEN: {MAX_SEQ_LEN}")

# =================================================================
# 2. 모델 재초기화 (수정된 사이즈 반영)
# =================================================================
N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2

transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=ACTUAL_VOCAB_SIZE, # ★ 수정됨
    tgt_vocab_size=ACTUAL_VOCAB_SIZE, # ★ 수정됨
    pos_len=MAX_SEQ_LEN,              # ★ 수정됨
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
lr_scheduler = CustomSchedule(D_MODEL) # 이전 코드의 클래스 재사용

# =================================================================
# 3. 데이터로더 및 훈련 루프
# =================================================================
BATCH_SIZE = 64
EPOCHS = 10

train_dataset = TensorDataset(torch.tensor(enc_train, dtype=torch.long), 
                              torch.tensor(dec_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("\n🚀 훈련을 다시 시작합니다...")

for epoch in range(EPOCHS):
    transformer.train()
    total_loss = 0
    
    for i, (src, tgt) in enumerate(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_real = tgt[:, 1:]
        
        # 마스크 생성 (이전 코드의 generate_masks 사용)
        enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_input)
        
        optimizer.zero_grad()
        
        output, _, _, _ = transformer(src, tgt_input, enc_mask, dec_enc_mask, dec_mask)
        
        # ★ VOCAB_SIZE 파라미터도 ACTUAL_VOCAB_SIZE로 변경
        loss = criterion(output.contiguous().view(-1, ACTUAL_VOCAB_SIZE), 
                         tgt_real.contiguous().view(-1))
        loss.backward()
        
        step = epoch * len(train_loader) + i + 1
        for pg in optimizer.param_groups:
            pg['lr'] = lr_scheduler(step)
            
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

📊 데이터 기준 안전한 VOCAB_SIZE: 6869
📊 데이터 기준 안전한 POS_LEN: 92

🚀 훈련을 다시 시작합니다...
Epoch 1/10, Loss: 3324.1996
Epoch 2/10, Loss: 1008.6811
Epoch 3/10, Loss: 507.4877
Epoch 4/10, Loss: 333.4148
Epoch 5/10, Loss: 256.0579
Epoch 6/10, Loss: 212.0195
Epoch 7/10, Loss: 182.8633
Epoch 8/10, Loss: 160.6883
Epoch 9/10, Loss: 143.9778
Epoch 10/10, Loss: 130.6699


In [68]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(reference, candidate, weights=[0.25, 0.25, 0.25, 0.25]):
    # reference: 정답 문장 토큰 리스트 (예: ['나', '도', '좋', '아'])
    # candidate: 모델이 예측한 토큰 리스트 (예: ['나', '도', '좋', '은데'])
    smoothie = SmoothingFunction().method1
    
    # BLEU Score 계산 (0.0 ~ 1.0)
    score = sentence_bleu([reference], candidate, weights=weights, smoothing_function=smoothie)
    return score

In [71]:
import torch

def evaluate(sentence, transformer, vocab, max_len=40):
    transformer.eval() # 평가 모드로 전환
    
    # =================================================================
    # 💡 핵심 수정: 사전에서 시작/종료 토큰을 안전하게 가져오기
    # '<start>', 'start', 심지어 없다면 기본값(1)을 사용하도록 안전망 구축
    # =================================================================
    START_TOKEN = vocab.get("<start>") or vocab.get("start") or vocab.get("<s>") or 1
    END_TOKEN = vocab.get("<end>") or vocab.get("end") or vocab.get("</s>") or 2
    
    # 1. 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    tokens = mecab.morphs(sentence)
    
    # 2. 정수 인코딩
    src_ids = [vocab.get(w, 0) for w in tokens]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)
    
    # 3. 디코더 시작 토큰 준비
    tgt_ids = [START_TOKEN]
    tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
    
    with torch.no_grad(): # 메모리 절약
        for _ in range(max_len):
            # 마스크 생성
            enc_mask, dec_enc_mask, dec_mask = generate_masks(src_tensor, tgt_tensor)
            
            # 예측
            output, _, _, _ = transformer(src_tensor, tgt_tensor, enc_mask, dec_enc_mask, dec_mask)
            
            # 다음 단어 인덱스 추출
            next_word_id = output[:, -1, :].argmax(dim=-1).item()
            
            # 타겟 텐서에 추가
            tgt_ids.append(next_word_id)
            tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
            
            # 종료 토큰을 만나면 멈춤
            if next_word_id == END_TOKEN:
                break
                
    # 4. 디코딩 (정수 -> 단어 복원)
    inv_vocab = {v: k for k, v in vocab.items()}
    
    # 시작 토큰과 종료 토큰은 제외하고 실제 단어만 반환
    decoded_words = [inv_vocab.get(i, "") for i in tgt_ids[1:] if i != END_TOKEN]
    
    return decoded_words

In [72]:
import random

# --- [1] 정성적 평가 (눈으로 확인하기) ---
print("==============================")
print(" 🤖 챗봇 대답 테스트")
print("==============================\n")

test_sentences = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

for s in test_sentences:
    # 대답 생성 후 리스트를 띄어쓰기로 연결하여 문장으로 만듦
    answer_tokens = evaluate(s, transformer, vocab)
    answer_sentence = " ".join(answer_tokens)
    print(f"Q: {s}")
    print(f"A: {answer_sentence}\n")

# --- [2] 정량적 평가 (BLEU Score 측정) ---
print("==============================")
print(" 📊 BLEU Score 평가 (샘플 100개)")
print("==============================\n")

# 원본 말뭉치(que_corpus, ans_corpus)에서 랜덤으로 100개 추출
sample_indices = random.sample(range(len(que_corpus)), 100)

total_bleu = 0

for idx in sample_indices:
    # 1. 질문의 원본 문장 복원 (정수 -> 단어)
    q_tokens = que_corpus[idx]
    q_sentence = " ".join(q_tokens) # 모델에 넣기 위해 문자열로 변환
    
    # 2. 실제 정답 토큰 (Reference)
    # <start>, <end>가 포함되지 않은 원본 토큰
    reference = ans_corpus[idx] 
    
    # 3. 모델의 예측 토큰 (Candidate)
    candidate = evaluate(q_sentence, transformer, vocab)
    
    # 4. BLEU 계산 및 누적
    bleu = calculate_bleu(reference, candidate)
    total_bleu += bleu

avg_bleu = total_bleu / 100
print(f"🔥 100개 샘플에 대한 평균 BLEU Score: {avg_bleu:.4f}")

# BLEU 점수 해석
if avg_bleu >= 0.3:
    print("평가: 매우 우수한 번역/대화 성능입니다! (보통 0.2~0.3 이상이면 대화 가능 수준)")
else:
    print("평가: 모델이 조금 더 학습해야 할 것 같습니다. (에포크를 늘리거나 데이터를 점검해보세요)")

 🤖 챗봇 대답 테스트

Q: 지루하다, 놀러가고 싶어.
A:       

Q: 오늘 일찍 일어났더니 피곤하다.
A:       

Q: 간만에 여자친구랑 데이트 하기로 했어.
A:       

Q: 집에 있는다는 소리야.
A:       

 📊 BLEU Score 평가 (샘플 100개)

🔥 100개 샘플에 대한 평균 BLEU Score: 0.0000
평가: 모델이 조금 더 학습해야 할 것 같습니다. (에포크를 늘리거나 데이터를 점검해보세요)


In [73]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# =================================================================
# 1. 에러 원인 완벽 차단: 절대 터지지 않는 VOCAB_SIZE와 POS_LEN 계산
# =================================================================
# 데이터에 존재하는 가장 큰 숫자(ID)를 찾아 무조건 그것보다 1 크게 만듭니다.
ACTUAL_VOCAB_SIZE = int(np.max([enc_train.max(), dec_train.max()])) + 1

# 문장 길이도 데이터셋의 최대 길이보다 무조건 길게 설정합니다.
MAX_SEQ_LEN = max(enc_train.shape[1], dec_train.shape[1]) + 50 

print(f"📊 데이터 기준 안전한 VOCAB_SIZE: {ACTUAL_VOCAB_SIZE}")
print(f"📊 데이터 기준 안전한 POS_LEN: {MAX_SEQ_LEN}")

# =================================================================
# 2. 모델 재초기화 (수정된 사이즈 반영)
# =================================================================
N_LAYERS = 1
D_MODEL = 368
N_HEADS = 8
D_FF = 1024
DROPOUT = 0.2

transformer = Transformer(
    n_layers=N_LAYERS,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    d_ff=D_FF,
    src_vocab_size=ACTUAL_VOCAB_SIZE, # ★ 수정됨
    tgt_vocab_size=ACTUAL_VOCAB_SIZE, # ★ 수정됨
    pos_len=MAX_SEQ_LEN,              # ★ 수정됨
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
lr_scheduler = CustomSchedule(D_MODEL) # 이전 코드의 클래스 재사용

# =================================================================
# 3. 데이터로더 및 훈련 루프
# =================================================================
BATCH_SIZE = 64
EPOCHS = 100

train_dataset = TensorDataset(torch.tensor(enc_train, dtype=torch.long), 
                              torch.tensor(dec_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("\n🚀 훈련을 다시 시작합니다...")

for epoch in range(EPOCHS):
    transformer.train()
    total_loss = 0
    
    for i, (src, tgt) in enumerate(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_real = tgt[:, 1:]
        
        # 마스크 생성 (이전 코드의 generate_masks 사용)
        enc_mask, dec_enc_mask, dec_mask = generate_masks(src, tgt_input)
        
        optimizer.zero_grad()
        
        output, _, _, _ = transformer(src, tgt_input, enc_mask, dec_enc_mask, dec_mask)
        
        # ★ VOCAB_SIZE 파라미터도 ACTUAL_VOCAB_SIZE로 변경
        loss = criterion(output.contiguous().view(-1, ACTUAL_VOCAB_SIZE), 
                         tgt_real.contiguous().view(-1))
        loss.backward()
        
        step = epoch * len(train_loader) + i + 1
        for pg in optimizer.param_groups:
            pg['lr'] = lr_scheduler(step)
            
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

📊 데이터 기준 안전한 VOCAB_SIZE: 6869
📊 데이터 기준 안전한 POS_LEN: 92

🚀 훈련을 다시 시작합니다...
Epoch 1/100, Loss: 3338.6072
Epoch 2/100, Loss: 1022.1082
Epoch 3/100, Loss: 517.9514
Epoch 4/100, Loss: 336.3885
Epoch 5/100, Loss: 257.4874
Epoch 6/100, Loss: 212.0071
Epoch 7/100, Loss: 182.2793
Epoch 8/100, Loss: 160.3778
Epoch 9/100, Loss: 143.8515
Epoch 10/100, Loss: 130.6806
Epoch 11/100, Loss: 120.1751
Epoch 12/100, Loss: 111.4926
Epoch 13/100, Loss: 103.5693
Epoch 14/100, Loss: 96.6586
Epoch 15/100, Loss: 90.6091
Epoch 16/100, Loss: 85.1211
Epoch 17/100, Loss: 80.4167
Epoch 18/100, Loss: 76.2685
Epoch 19/100, Loss: 72.8610
Epoch 20/100, Loss: 69.2195
Epoch 21/100, Loss: 65.9541
Epoch 22/100, Loss: 63.2692
Epoch 23/100, Loss: 60.2232
Epoch 24/100, Loss: 57.8329
Epoch 25/100, Loss: 55.7119
Epoch 26/100, Loss: 53.4956
Epoch 27/100, Loss: 51.7247
Epoch 28/100, Loss: 49.6802
Epoch 29/100, Loss: 48.0240
Epoch 30/100, Loss: 46.5043
Epoch 31/100, Loss: 45.0893
Epoch 32/100, Loss: 43.4940
Epoch 33/100, Loss: 42.1

In [75]:
# 1. 번역(추론) 함수 정의
def evaluate(sentence, transformer, src_tokenizer, tgt_tokenizer, device):
    transformer.eval() # 모델을 평가 모드로 전환 (Dropout 비활성화)

    # 입력 문장 전처리 및 토큰화
    sentence = preprocess_sentence(sentence)
    pieces = src_tokenizer.encode_as_pieces(sentence)
    tokens = [src_tokenizer.piece_to_id(p) for p in pieces]
    input_tensor = torch.tensor([tokens], dtype=torch.long).to(device)

    # 인코더 마스크 생성
    enc_mask = (input_tensor != 0).unsqueeze(1).unsqueeze(2)

    # 인코더 통과
    with torch.no_grad():
        enc_output = transformer.encoder(input_tensor, enc_mask)

    # 디코더 입력 시작: <sos> 토큰 (SentencePiece의 경우 보통 ID 1 또는 2, 여기서는 1로 가정)
    output_tokens = [src_tokenizer.bos_id()] # <sos> ID
    
    # 최대 길이에 도달할 때까지 한 단어씩 예측 (Greedy Search)
    for i in range(MAX_SEQ_LEN):
        tgt_input = torch.tensor([output_tokens], dtype=torch.long).to(device)
        
        # 디코더 마스크 생성 (Look-ahead mask 포함)
        _, dec_enc_mask, dec_mask = generate_masks(input_tensor, tgt_input)
        
        # 디코더 통과 및 예측
        with torch.no_grad():
            predictions, _, _, _ = transformer.decoder(tgt_input, enc_output, enc_mask, dec_enc_mask, dec_mask)
        
        # 마지막 타임스텝의 단어 선택
        prediction = predictions[:, -1:, :].argmax(dim=-1).item()
        
        # <eos> 토큰이 나오면 멈춤
        if prediction == src_tokenizer.eos_id():
            break
            
        output_tokens.append(prediction)

    # 토큰을 다시 문장으로 변환
    result = tgt_tokenizer.decode_ids(output_tokens[1:]) # <sos> 제외하고 디코딩
    return result

# 2. 테스트 데이터셋에 대해 BLEU Score 계산
def calculate_bleu(test_src_sentences, test_tgt_sentences, transformer, src_tokenizer, tgt_tokenizer, device):
    total_bleu = 0
    cnt = 0
    
    print("\n🔍 테스트 데이터 번역 및 평가 중...")
    for src_sen, tgt_sen in tqdm(zip(test_src_sentences, test_tgt_sentences), total=len(test_src_sentences)):
        pred_sen = evaluate(src_sen, transformer, src_tokenizer, tgt_tokenizer, device)
        
        # BLEU 계산을 위해 단어 단위로 쪼갬
        reference = [tgt_sen.split()]
        candidate = pred_sen.split()
        
        # SmoothingFunction을 사용하여 짧은 문장에서도 점수 보정
        score = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
        total_bleu += score
        cnt += 1

    avg_bleu = (total_bleu / cnt) * 100
    print(f"\n✅ 최종 평균 BLEU Score: {avg_bleu:.2f}")
    return avg_bleu


# 

print("슝=3")

슝=3


In [78]:
import sentencepiece as spm

# 1. 데이터를 텍스트 파일로 임시 저장 (SentencePiece 학습용)
def temp_file_for_spm(sentences, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for sen in sentences:
            f.write(sen + '\n')

temp_file_for_spm(train_eng_sentences, 'eng.txt')
temp_file_for_spm(train_spa_sentences, 'spa.txt')

# 2. 영어 통역사(sp_enc) 학습 및 로드
spm.SentencePieceTrainer.Train(
    '--input=eng.txt --model_prefix=eng_spm --vocab_size=8000'
)
sp_enc = spm.SentencePieceProcessor()
sp_enc.Load('eng_spm.model')

# 3. 스페인어 통역사(sp_dec) 학습 및 로드
spm.SentencePieceTrainer.Train(
    '--input=spa.txt --model_prefix=spa_spm --vocab_size=8000'
)
sp_dec = spm.SentencePieceProcessor()
sp_dec.Load('spa_spm.model')

print("✅ 통역사(sp_enc, sp_dec) 준비 완료!")

✅ 통역사(sp_enc, sp_dec) 준비 완료!


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=eng.txt --model_prefix=eng_spm --vocab_size=8000
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: eng.txt
  input_format: 
  model_prefix: eng_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos

In [81]:
def chatbot_answer(sentence, transformer, tokenizer, device, max_len=40):
    transformer.eval()

    # 1. 입력 문장 전처리 및 정수 인코딩
    sentence = preprocess_sentence(sentence)
    # Keras Tokenizer를 사용하여 정수로 변환
    sequence = tokenizer.texts_to_sequences([sentence])[0]
    # 모델에 맞는 길이로 패딩 (학습 때와 동일하게)
    input_tensor = torch.tensor([sequence], dtype=torch.long).to(device)

    # 2. 시작/끝 토큰 ID 확인
    start_id = tokenizer.word_index.get('<start>')
    end_id = tokenizer.word_index.get('<end>')

    # 3. 디코더 입력 시작 (<start>)
    output_tokens = [start_id]
    
    for i in range(max_len):
        tgt_input = torch.tensor([output_tokens], dtype=torch.long).to(device)
        
        # 마스크 생성 (기존 generate_masks 함수 사용)
        enc_mask, dec_enc_mask, dec_mask = generate_masks(input_tensor, tgt_input)
        
        with torch.no_grad():
            output, _, _, _ = transformer(input_tensor, tgt_input, enc_mask, dec_enc_mask, dec_mask)
        
        # 가장 높은 확률의 단어 ID 선택
        prediction = output[:, -1:, :].argmax(dim=-1).item()
        
        # <end> 토큰이 나오면 멈춤
        if prediction == end_id:
            break
            
        output_tokens.append(prediction)

    # 4. 정수 리스트를 다시 한글 문장으로 변환
    # <start>와 <end> 제외하고 복원
    result_list = [tok for tok in output_tokens if tok != start_id and tok != end_id]
    result = tokenizer.sequences_to_texts([result_list])[0]
    
    return result

# --- 🚀 과제 예문 출력 ---
test_questions = [
    "지루하다, 놀러가고 싶어.",
    "오늘 일찍 일어났더니 피곤하다.",
    "간만에 여자친구랑 데이트 하기로 했어.",
    "집에 있는다는 소리야."
]

print("✨ 한국어 챗봇 답변 결과 ✨\n")
for q in test_questions:
    ans = chatbot_answer(q, transformer, tokenizer, device)
    print(f"Q: {q}")
    print(f"A: {ans} <end>")
    print("-" * 30)

✨ 한국어 챗봇 답변 결과 ✨

Q: 지루하다, 놀러가고 싶어.
A:  <end>
------------------------------
Q: 오늘 일찍 일어났더니 피곤하다.
A: 맛있 게 쉬 어 보 세요 . <end>
------------------------------
Q: 간만에 여자친구랑 데이트 하기로 했어.
A: 좋 은 해도 돼요 . <end>
------------------------------
Q: 집에 있는다는 소리야.
A:  <end>
------------------------------


In [82]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def calculate_bleu(test_questions, test_answers, transformer, tokenizer, device):
    total_score = 0
    cnt = 0
    
    # 점수 보정 함수 (짧은 문장에서 0점이 나오는 것을 방지)
    smoothing = SmoothingFunction().method1
    
    print("📢 테스트 데이터로 BLEU 점수를 측정합니다...")
    
    for q, a in tqdm(zip(test_questions, test_answers), total=len(test_questions)):
        # 1. 모델이 예측한 답변 생성
        # (앞서 만든 chatbot_answer 함수를 활용합니다)
        pred_ans = chatbot_answer(q, transformer, tokenizer, device)
        
        # 2. BLEU 계산을 위한 토큰화 (Mecab 형태소 단위로 비교하는 것이 정확합니다)
        # 만약 이미 토큰화된 리스트라면 바로 사용하고, 아니면 다시 쪼갭니다.
        reference = [mecab.morphs(preprocess_sentence(a))] # 정답 (리스트의 리스트 형태여야 함)
        candidate = mecab.morphs(preprocess_sentence(pred_ans)) # 모델의 대답
        
        # 3. BLEU 점수 계산
        score = sentence_bleu(reference, candidate, smoothing_function=smoothing)
        
        total_score += score
        cnt += 1
    
    avg_bleu = (total_score / cnt) * 100
    print(f"\n✅ 최종 평균 BLEU Score: {avg_bleu:.4f}")
    
    return avg_bleu

# --- 실행 부분 ---
# test_questions와 test_answers는 훈련에 쓰이지 않은 별도의 테스트용 리스트여야 합니다.
# 만약 없다면 전체 데이터 중 일부를 슬라이싱해서 넣어보세요.
bleu_result = calculate_bleu(questions[-100:], answers[-100:], transformer, tokenizer, device)

📢 테스트 데이터로 BLEU 점수를 측정합니다...


100%|██████████| 100/100 [00:03<00:00, 27.89it/s]


✅ 최종 평균 BLEU Score: 3.4884


결과 및 분석
최종 BLEU Score: 3.4884

평가: 수치는 낮으나, 정답이 고정된 챗봇 데이터 특성상 문장 구조를 파악하고 기본 답변을 생성하는 수준에 도달함.(정성적으로 나쁘지 않아보임.)